
# Phase 3C.03 — Cross-Asset & Lookback Robustness

**Objective:** Evaluate the robustness of Time-Series Momentum across 16 ETFs, four asset classes, and predefined lookbacks (3, 6, 12 months).

The 12-month lookback remains the primary specification. Analysis is restricted to Train (2010–2017) and Validation (2018–2021). Test data remain excluded.


## 1. Environment and Data Preparation

### 1.1. Imports and Configuration

Configure the Python environment and import reusable functions for data preparation, signal construction, and validation.


In [70]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import importlib
import src.data
import src.signals

importlib.reload(src.data)
importlib.reload(src.signals)

from src.data import download_prices, to_monthly_prices
from src.signals import calculate_momentum, generate_signals, clean_signals_and_targets, generate_targets
from src.data import validate_monthly_prices


### 1.2. Data Loading and Integrity Validation

Download and resample adjusted prices for the 16 ETFs over 2008–2021.

Validate data integrity using the reusable `validate_monthly_prices()` function, checking asset coverage, missing values, duplicate dates, chronological order, and monthly continuity.

Invalid datasets must stop the research pipeline.


In [71]:
asset_universe = {
    "Equities" : ["SPY", "VGK", "EWJ", "EEM"],
    "Commodities" : ["USO", "DBA", "DBB", "DBC"],
    "Bonds" : ["IEF", "TLT", "LQD", "HYG"],
    "Forex" : ["FXE", "FXY", "FXB", "FXA"]
}

tickers = []

for assets in asset_universe.values():
    tickers.extend(assets)

prices = download_prices(tickers, "2008-01-01", "2022-01-01", auto_adjust=True, progress=False)

monthly_prices = to_monthly_prices(prices)

validate_monthly_prices(monthly_prices, tickers, "2008-01-31", "2021-12-31")
print("Data validation passed")

Data validation passed



### 1.3. Signal and Target Preparation

Construct time-series momentum signals for the 16 ETFs using **3-, 6-, and 12-month lookback windows**. The 12-month lookback remains the primary specification, while 3 and 6 months are used for robustness analysis. All signals predict the direction of the **next-month return** (1-month prediction horizon).

Split signals and targets according to signal formation dates:
- **Train:** December 2009 – November 2017 (96 months).
- **Validation:** December 2017 – November 2021 (48 months).
- **Test:** Excluded from this phase.

For each lookback, apply `clean_signals_and_targets()` to identify valid signal-target pairs, excluding missing and neutral observations. The primary 12-month specification retains **1,531 Train** and **765 Validation** observations.

For lookback robustness comparisons, construct a **common validity mask** requiring each asset-month observation to be valid across all three lookbacks. Apply this mask to signals and targets using `.where()` to ensure identical evaluation samples.

**Final common sample:**
- **Train:** 1,529 valid observations.
- **Validation:** 762 valid observations.

The cross-asset analysis uses the individual 12-month validity mask, while lookback comparisons use the common sample to eliminate differences caused by sample composition.


In [72]:
momentum_12m = calculate_momentum(monthly_prices, lookback=12)
signals_12m = generate_signals(momentum_12m)

momentum_6m = calculate_momentum(monthly_prices, lookback=6)
signals_6m = generate_signals(momentum_6m)

momentum_3m = calculate_momentum(monthly_prices, lookback=3)
signals_3m = generate_signals(momentum_3m)

targets_1m = generate_targets(monthly_prices, horizon=1)

signals_by_lookback = {
    3: signals_3m,
    6: signals_6m,
    12: signals_12m
}

train_signals_by_lookback = {}
validation_signals_by_lookback = {}

for lookback, signals in signals_by_lookback.items():
    train_lookback = signals.loc["2009-12-31":"2017-11-30"]
    validation_lookback = signals.loc["2017-12-31":"2021-11-30"]

    train_signals_by_lookback[lookback] = train_lookback
    validation_signals_by_lookback[lookback] = validation_lookback

train_targets = targets_1m.loc["2009-12-31":"2017-11-30"]
validation_targets = targets_1m.loc["2017-12-31":"2021-11-30"]

train_mask = {}
validation_mask = {}

for lookback, signals in train_signals_by_lookback.items():
    _, _, mask = clean_signals_and_targets(signals, train_targets)

    train_mask[lookback] = mask

    print(f"Train {lookback}M: {signals.shape}")
    print(f"Train {lookback}M mask: {train_mask[lookback].shape}")
    print(f"Valid observations ({lookback}M): {mask.sum().sum()}")
    print("")


print("\n")

for lookback, signals in validation_signals_by_lookback.items():
    _, _, mask = clean_signals_and_targets(signals, validation_targets)

    validation_mask[lookback] = mask

    print(f"Validation {lookback}M: {signals.shape}")
    print(f"Validation {lookback}M mask: {validation_mask[lookback].shape}")
    print(f"Valid observations ({lookback}M): {mask.sum().sum()}")
    print("")

print("\n")

print("Train targets:", train_targets.shape)
print("Validation targets:", validation_targets.shape)

train_common_mask = train_mask[3] & train_mask[6] & train_mask[12]
validation_common_mask = validation_mask[3] & validation_mask[6] & validation_mask[12]

print("\n")

print(f"Train common mask : {train_common_mask.sum().sum()}")
print(f"Validation common mask : {validation_common_mask.sum().sum()}")

train_signals_common = {}
validation_signals_common = {}

print("\n")

for lookback, signals in train_signals_by_lookback.items():
    train_signals_common[lookback] = train_signals_by_lookback[lookback].where(train_common_mask)
    print(f"Train {lookback}M common signals: {train_signals_common[lookback].shape}")

print("")

for lookback, signals in validation_signals_by_lookback.items():
    validation_signals_common[lookback] = validation_signals_by_lookback[lookback].where(validation_common_mask)
    print(f"Validation {lookback}M common signals: {validation_signals_common[lookback].shape}")

print("\n")

train_targets_common = train_targets.where(train_common_mask)
validation_targets_common = validation_targets.where(validation_common_mask)

for lookback, signals in train_signals_common.items():
    print(f"Train {lookback}M: {signals.notna().sum().sum()}")

print("")

for lookback, signals in validation_signals_common.items():
    print(f"Validation {lookback}M: {signals.notna().sum().sum()}")

print("\n")

print(train_targets_common.notna().sum().sum())
print(validation_targets_common.notna().sum().sum())

Train 3M: (96, 16)
Train 3M mask: (96, 16)
Valid observations (3M): 1530

Train 6M: (96, 16)
Train 6M mask: (96, 16)
Valid observations (6M): 1532

Train 12M: (96, 16)
Train 12M mask: (96, 16)
Valid observations (12M): 1531



Validation 3M: (48, 16)
Validation 3M mask: (48, 16)
Valid observations (3M): 763

Validation 6M: (48, 16)
Validation 6M mask: (48, 16)
Valid observations (6M): 764

Validation 12M: (48, 16)
Validation 12M mask: (48, 16)
Valid observations (12M): 765



Train targets: (96, 16)
Validation targets: (48, 16)


Train common mask : 1529
Validation common mask : 762


Train 3M common signals: (96, 16)
Train 6M common signals: (96, 16)
Train 12M common signals: (96, 16)

Validation 3M common signals: (48, 16)
Validation 6M common signals: (48, 16)
Validation 12M common signals: (48, 16)


Train 3M: 1529
Train 6M: 1529
Train 12M: 1529

Validation 3M: 762
Validation 6M: 762
Validation 12M: 762


1529
762



## 2. Cross-Asset Predictive Analysis

Evaluate the predictive performance of the primary 12-month momentum signal for each of the 16 ETFs.

Compare Train and Validation results to identify cross-asset dispersion and potential instability in predictive accuracy.

### 2.1. Individual Asset Accuracy
